# Fingerprint authentication experiment

This notebook is a presentation and inspection layer over the reusable `experiments` package. It loads the artifacts generated by `python -m experiments --modality fingerprint`. The cells below are intentionally separated by artifact type so the complete authentication experiment can be inspected without manually browsing the results tree.

In [ ]:
from pathlib import Path
from PIL import Image as PILImage
import matplotlib.pyplot as plt
from IPython.display import Image, display

from pathlib import Path
import json
import numpy as np
import pandas as pd

from experiments.analysis import rank_modality_trials
from experiments.io import load_embedding_artifact
from experiments.protocols import PROTOCOLS
import config

ROOT = Path.cwd()
RESULTS = ROOT / "results" / "experiments" / "fingerprint"

manifest = json.loads((RESULTS / "manifest.json").read_text())
manifest

## 1. Metrics and operating points

In [ ]:
# Same metrics as CSV, exposed separately for convenient filtering/sorting.
metrics_df = pd.read_csv(RESULTS / "metrics.csv")
print("=== Fingerprint Authentication metrics ===")
display(metrics_df.sort_values(["model_name"]))


## 2. Score distributions stored by the experiment

In [ ]:
score_rows = []
for path in sorted((RESULTS / "scores").glob("*.npz")):
    with np.load(path) as data:
        score_rows.append({
            "artifact": path.name,
            "genuine_trials": int(data["genuine"].size),
            "impostor_trials": int(data["impostor"].size),
            "genuine_mean": float(data["genuine"].mean()),
            "impostor_mean": float(data["impostor"].mean()),
        })
display(pd.DataFrame(score_rows))

## 3. All generated comparison and authentication plots

In [ ]:
headings = {
    "comparisons": "Comparisions Plots", 
    "det": "DET (Detection Error Tradeoff Plots)", 
    "distributions": "Score Distributions Plot", 
    "far_frr": "FAR_FRR_vs_Threshold Plots", 
    "pca": "PCA Plots", 
    "roc": "ROC Curve"
}

def show_plot_group(group, width=600):
    folder = RESULTS / "plots" / group
    print(f"\n### {headings.get(group, None)}")
    if not folder.exists():
        print("Not generated yet.")
        return
    for path in sorted(folder.glob("*.png")):
        print(path.name)
        display(Image(filename=str(path), width=width))


for group in ["comparisons", "det", "distributions", "far_frr", "pca", "roc"]:
    show_plot_group(group)

## 4. Genuine-trial analysis — highest genuine scores

This cell ranks all genuine comparisons by score. Set `N` in the next cell. Each row shows a representative enrollment sample, the verification sample, both subject IDs, and the model score. If an enrollment template was created by averaging multiple samples, the full list of contributing paths is retained in the table data.

In [ ]:
# Trial-analysis display helper. N is intentionally configurable in this cell.
N = 10

def show_trial_gallery(
    rows,
    title,
    max_enrollment_images,
):
    """Display every source image represented by each ranked trial.

    The experiment metadata may contain multiple enrollment and/or verification
    images for one trial (for example, 3 enrollment + 2 verification images
    when subject-level templates are used). Each trial is rendered as its own
    image block so no source image is silently discarded. Metadata is placed
    below the images instead of being drawn over them.
    """
    rows = rows[:N]

    if not rows:
        print("No rows available.")
        return

    for row_index, row in enumerate(rows, start=1):
        enrollment_paths = row.get("enrollment_paths", [])

        # Support both the new list format and the older single-path format.
        if not enrollment_paths:
            enrollment_path = row.get("enrollment_path")
            if enrollment_path:
                enrollment_paths = [enrollment_path]

        verification_paths = row.get("verification_paths", [])

        # Support older metadata containing only verification_path.
        if not verification_paths:
            verification_path = row.get("verification_path")
            if verification_path:
                verification_paths = [verification_path]

        # Apply user-configurable limits.
        enrollment_paths = enrollment_paths[:max_enrollment_images]
        verification_paths = verification_paths[:1]

        # Number of columns is the maximum number of images in either row.
        n_columns = max(
            len(enrollment_paths),
            len(verification_paths),
            1,
        )

        # Two image rows: enrollment + verification.
        fig, axes = plt.subplots(
            2,
            n_columns,
            figsize=(3.2 * n_columns, 7.0),
            squeeze=False,
        )

        fig.suptitle(
            f"Trial {row_index}: "
            f"Enrollment subject = {row['enrollment_key']} | "
            f"Verification subject = {row['verification_key']} | "
            f"Score = {row['score']:.4f}",
            fontsize=12,
            y=0.98,
        )

        def display_images(paths, row_idx, label):
            for col_idx in range(n_columns):
                axis = axes[row_idx][col_idx]
                axis.axis("off")

                if col_idx >= len(paths):
                    continue

                path = paths[col_idx]

                try:
                    image = PILImage.open(path).convert("RGB")
                    axis.imshow(image)
                    axis.set_title(
                        f"{label} {col_idx + 1}",
                        fontsize=9,
                    )
                except Exception as exc:
                    axis.text(
                        0.5,
                        0.5,
                        f"Unable to load image\n{exc}",
                        ha="center",
                        va="center",
                        fontsize=8,
                    )

        display_images(
            enrollment_paths,
            0,
            "Enrollment",
        )

        display_images(
            verification_paths,
            1,
            "Verification",
        )

        # Metadata is placed BELOW the images rather than on top of them.
        metadata = (
            f"Enrollment subject: {row['enrollment_key']}    |    "
            f"Verification subject: {row['verification_key']}    |    "
            f"Score: {row['score']:.4f}\n"
            f"Showing {len(enrollment_paths)} enrollment sample(s) "
            f"and {len(verification_paths)} verification sample(s)"
        )

        fig.text(
            0.5,
            0.015,
            metadata,
            ha="center",
            va="bottom",
            fontsize=10,
        )

        # Leave enough space for the metadata at the bottom.
        plt.tight_layout(
            rect=(0, 0.08, 1, 0.94)
        )

        plt.show()

# Select the model whose trials you want to inspect.
MODEL_FOR_TRIAL_ANALYSIS = metrics_df.iloc[0]["model_name"]
data = load_embedding_artifact("fingerprint", MODEL_FOR_TRIAL_ANALYSIS)
genuine_top, _ = rank_modality_trials(
    data,
    PROTOCOLS["fingerprint"],
    instance_level_enrollment=config.STANDALONE_VIRTUAL_ENROLLMENT,
    instance_level_verification=config.STANDALONE_VIRTUAL_VERIFICATION,
    top_n=N,
)
display(pd.DataFrame([{key: value for key, value in row.items() if key not in {"enrollment_paths", "verification_paths"}} for row in genuine_top]))
show_trial_gallery(genuine_top, title=f"Top {{N}} genuine trials — {{MODEL_FOR_TRIAL_ANALYSIS}}", max_enrollment_images=2)


## 5. Impostor-trial analysis — highest impostor scores

These are the hardest impostor comparisons: high impostor similarity means the model found a wrong subject that looks unusually similar to the verification sample.

In [ ]:
MODEL_FOR_TRIAL_ANALYSIS = metrics_df.iloc[0]["model_name"]
N = 10  # Change this value to inspect more/fewer top hard impostors.
data = load_embedding_artifact("fingerprint", MODEL_FOR_TRIAL_ANALYSIS)
_, impostor_top = rank_modality_trials(
    data,
    PROTOCOLS["fingerprint"],
    instance_level_enrollment=config.STANDALONE_VIRTUAL_ENROLLMENT,
    instance_level_verification=config.STANDALONE_VIRTUAL_VERIFICATION,
    top_n=N,
)
display(pd.DataFrame([{key:value for key,value in row.items() if key != "enrollment_paths"} for row in impostor_top]))
show_trial_gallery(impostor_top, f"Top {N} hard impostor trials — {MODEL_FOR_TRIAL_ANALYSIS}", max_enrollment_images=2)
